## Prediction using Support Vector Machines

### Why SVM?

We have explored the dataset and clearly its a classification problem. The reason this algorithm is explored because :-


1.   Not a very large dataset - few ten thousand rows
2.   Dataset doesnot have any missing values
3.   High dimensional data - upto 30 feature columns
4.   C parameter prevents overfitting
5.   From EDA



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# importing dataset

data = pd.read_csv('/content/drive/MyDrive/Credit Card Fraud Colab Folder/credit_card_fraud.csv')

In [3]:
data['fraud'] = data['fraud'].replace({'fraud': 1, 'otherwise': 0})

# standard train_test_split doesnot always produces train data with fraud == 1

# manually making train data and test data

data_train_fraud = data[data['fraud']==1].head(36)
data_test_fraud = data[data['fraud']==1].tail(14)

data_not_fraud = data[data['fraud']==0]

data_train_not_fraud = data_not_fraud.sample(frac=0.8)
data_test_not_fraud = data_not_fraud[~data_not_fraud.index.isin(data_train_not_fraud.index)]

data_train = pd.concat([data_train_fraud,data_train_not_fraud])
data_test = pd.concat([data_test_fraud,data_test_not_fraud])

X_test = data_test.drop('fraud', axis=1)
y_test = data_test['fraud']
X_train = data_train.drop('fraud', axis=1)
y_train = data_train['fraud']

# scaling data

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_train)
X_scaled_test = scaler.fit_transform(X_test)

/tmp/ipykernel_2636/2291736659.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['fraud'] = data['fraud'].replace({'fraud': 1, 'otherwise': 0})


In [4]:
# preparing model

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# values to search across for

param_grid = {'kernel' : ['poly','rbf','linear'],
              'C' : np.linspace(0.001,1,100),
              #'epsilon' : [0.01, 0.05, 0.1, 0.2, 0.5]
}

# base model

base_model = SVC(class_weight={0:1, 1:10})      # this class_weight tells about class imbalance

# grid model

grid_model = GridSearchCV(base_model, param_grid)

In [20]:
# fitting the grid model to data

grid_model.fit(X_scaled_train, y_train)

GridSearchCV(estimator=SVC(class_weight={0: 1, 1: 10}),
             param_grid={'C': array([0.001     , 0.01109091, 0.02118182, 0.03127273, 0.04136364,
       0.05145455, 0.06154545, 0.07163636, 0.08172727, 0.09181818,
       0.10190909, 0.112     , 0.12209091, 0.13218182, 0.14227273,
       0.15236364, 0.16245455, 0.17254545, 0.18263636, 0.19272727,
       0.20281818, 0.21290909, 0.223     , 0.23309091, 0.24318182,
       0.25327273, 0.26...
       0.65690909, 0.667     , 0.67709091, 0.68718182, 0.69727273,
       0.70736364, 0.71745455, 0.72754545, 0.73763636, 0.74772727,
       0.75781818, 0.76790909, 0.778     , 0.78809091, 0.79818182,
       0.80827273, 0.81836364, 0.82845455, 0.83854545, 0.84863636,
       0.85872727, 0.86881818, 0.87890909, 0.889     , 0.89909091,
       0.90918182, 0.91927273, 0.92936364, 0.93945455, 0.94954545,
       0.95963636, 0.96972727, 0.97981818, 0.98990909, 1.        ]),
                         'kernel': ['poly', 'rbf', 'linear']})

In [21]:
# getting the best

grid_model.best_params_

{'C': np.float64(0.001), 'kernel': 'linear'}

In [20]:
# lets test the model

model = SVC(C=0.001, kernel='linear',class_weight={0:1, 1:100})
model.fit(X_scaled_train, y_train)

SVC(C=0.001, class_weight={0: 1, 1: 100}, kernel='linear')

In [21]:
yHat = model.predict(X_scaled_test)
yHat

array([1, 1, 1, ..., 0, 0, 0])

In [22]:
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
print(confusion_matrix(y_test,yHat))
print(classification_report(y_test,yHat))
print(accuracy_score(y_test,yHat))

[[5682    4]
 [   2   12]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5686
           1       0.75      0.86      0.80        14

    accuracy                           1.00      5700
   macro avg       0.87      0.93      0.90      5700
weighted avg       1.00      1.00      1.00      5700

0.9989473684210526


In [23]:
#final test with all data

y= data['fraud']
X = data.drop('fraud', axis=1)
X_scaled = scaler.fit_transform(X)

yHat = model.predict(X_scaled)
yHat


array([0, 0, 0, ..., 0, 0, 0])

In [24]:
print(confusion_matrix(y,yHat))
print(classification_report(y,yHat))
print(accuracy_score(y,yHat))

[[28401    29]
 [    7    42]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28430
           1       0.59      0.86      0.70        49

    accuracy                           1.00     28479
   macro avg       0.80      0.93      0.85     28479
weighted avg       1.00      1.00      1.00     28479

0.9987359106710207
